# Scikit-Learn — The Complete Guide to Classical Machine Learning

## What is Scikit-Learn?

Scikit-Learn (often written `sklearn`) is Python's **premier machine learning library** — the industry standard for classical (non-deep learning) ML. It provides consistent, well-documented implementations of virtually every classical algorithm: linear regression, random forests, SVMs, k-means clustering, PCA, and hundreds more.

**Real-world analogy**: If machine learning is cooking, scikit-learn is a professional kitchen with every tool you need — already sharpened, perfectly calibrated, and arranged so you can work fast. You don't have to invent the tools; you focus on the recipe.

## Why Scikit-Learn?

- **Consistent API**: every model has `.fit()`, `.predict()`, `.score()` — learn one, know them all
- **Production-ready**: used by Netflix, Spotify, Booking.com, and thousands of companies
- **Integrated pipeline**: preprocessing → model → evaluation in one object
- **Best for tabular data**: structured data (tables, CSVs) — this is most real-world ML
- **Strong community**: 50k+ stars on GitHub, extensive examples, 15+ years of development

## When to Use Sklearn vs Deep Learning

| Situation | Use Sklearn | Use Deep Learning |
|-----------|------------|------------------|
| Tabular data (< 1M rows) | ✓ | Usually overkill |
| Interpretability required | ✓ | Harder |
| Images, Audio, Text | ✗ | ✓ |
| Limited compute/time | ✓ | ✗ |
| Need uncertainty estimates | ✓ | Harder |
| Very large datasets | Limited | ✓ |

## Prerequisites
- Python basics
- NumPy arrays
- Basic statistics (mean, variance)

## Table of Contents
1. Installation & The Sklearn API Contract
2. Data Preprocessing (Scaling, Encoding, Imputation)
3. Supervised Learning: Regression
4. Supervised Learning: Classification
5. Model Evaluation (Metrics, Cross-Validation)
6. Ensemble Methods (Random Forest, Gradient Boosting)
7. Unsupervised Learning (Clustering, PCA)
8. Pipelines: The Professional Pattern
9. Hyperparameter Tuning (GridSearch, RandomSearch)
10. Feature Selection & Importance
11. Common Pitfalls
12. Mini Project: End-to-End ML Pipeline on Real Data
13. Interview Q&A
14. Resources

---

**Official Docs**: https://scikit-learn.org/stable/  
**User Guide**: https://scikit-learn.org/stable/user_guide.html  
**YouTube (StatQuest)**: https://www.youtube.com/c/joshstarmer (best ML theory explanations)  
**YouTube (Corey Schafer)**: https://www.youtube.com/playlist?list=PL-osiE80TeTd2vbCRGJrRhIiSEIYPF_Ij  
**Textbook (free)**: https://www.statlearning.com/ (Introduction to Statistical Learning)

## 1. Installation & The Sklearn API Contract

```bash
pip install scikit-learn numpy pandas matplotlib seaborn
```

### The One API Rule

**Every** sklearn estimator (model, transformer, scaler) follows the same API:

| Method | Purpose | Returns |
|--------|---------|--------|
| `estimator.fit(X, y)` | Learn from training data | `self` (the fitted estimator) |
| `estimator.predict(X)` | Make predictions | Array of predictions |
| `estimator.transform(X)` | Transform data (scalers, PCA) | Transformed array |
| `estimator.fit_transform(X)` | `fit` + `transform` in one step | Transformed array |
| `estimator.score(X, y)` | Model performance metric | Float (accuracy, R²) |

**This uniformity** means you can swap any model in a pipeline without changing other code!

In [ ]:
scikit-learntry:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns
    import sklearn
    from sklearn.datasets import load_iris, load_breast_cancer, make_classification
    from sklearn.model_selection import train_test_split, cross_val_score
    from sklearn.preprocessing import StandardScaler
    from sklearn.pipeline import Pipeline
    from sklearn.metrics import classification_report, roc_auc_score
    import warnings; warnings.filterwarnings("ignore")
    print(f"scikit-learn {sklearn.__version__} ready")
except ImportError:
    raise SystemExit("Run: pip install scikit-learn pandas numpy matplotlib seaborn")


---
## 2. Data Preprocessing

Raw data is almost never ready for ML. Preprocessing is often **more important than the model choice**.

### The Golden Rule
> **Always fit preprocessing on TRAINING data only. Then apply (transform) to both train AND test.**

**Why?** Fitting on all data (including test) = **data leakage** — the model indirectly sees test information during training. This causes falsely optimistic evaluation scores.

**Analogy**: You're studying for an exam. Data leakage is like studying the answers to the exam questions themselves — your practice score looks great, but you didn't actually learn.

In [ ]:
from sklearn.preprocessing import (
    StandardScaler,      # Mean=0, Std=1
    MinMaxScaler,        # Scale to [0, 1]
    RobustScaler,        # Robust to outliers (uses median/IQR)
    LabelEncoder,        # Integer encode labels
    OneHotEncoder,       # One-hot encode categorical features
    OrdinalEncoder,      # Ordinal encode ordered categories
)
from sklearn.impute import SimpleImputer

np.random.seed(42)

# ── Scaling ────────────────────────────────────────────────────────────────────
# Example: two features with very different scales
X_raw = np.array([[100, 0.001],
                   [150, 0.002],
                   [80,  0.0015],
                   [200, 0.003]])

X_train_raw = X_raw[:3]
X_test_raw  = X_raw[3:]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)  # fit on train!
X_test_scaled  = scaler.transform(X_test_raw)        # only transform test!

print("=== Scaling ===")
print(f"Raw train:\n{X_train_raw}")
print(f"Scaled train:\n{X_train_scaled.round(4)}")
print(f"Scaler learned: mean={scaler.mean_}, std={scaler.scale_.round(4)}")

# ── Encoding Categorical Variables ─────────────────────────────────────────────
# One-Hot Encoding: 'Color' → [is_Red, is_Blue, is_Green]
X_cat = pd.DataFrame({'Color': ['Red', 'Blue', 'Green', 'Red', 'Blue']})

ohe = OneHotEncoder(sparse_output=False, drop='first')  # drop='first' prevents multicollinearity
X_encoded = ohe.fit_transform(X_cat)
print(f"\n=== One-Hot Encoding ===")
print(f"Original: {X_cat['Color'].values}")
print(f"Encoded categories: {ohe.categories_[0]}")
print(f"Encoded (drop='first'):\n{X_encoded}")

# ── Missing Value Imputation ───────────────────────────────────────────────────
X_missing = np.array([[1, 2, np.nan],
                        [4, np.nan, 6],
                        [7, 8, 9],
                        [np.nan, 11, 12]])

imputer = SimpleImputer(strategy='mean')  # strategy: 'mean', 'median', 'most_frequent', 'constant'
X_imputed = imputer.fit_transform(X_missing)
print(f"\n=== Imputation ===")
print(f"With missing values:\n{X_missing}")
print(f"After mean imputation:\n{X_imputed}")

---
## 3. Supervised Learning: Regression

**Regression** predicts a **continuous number** (price, temperature, salary).

**Analogy**: If you're trying to predict the price of a house, that's regression. The answer is a specific number ($350,000), not a category.

### Key Algorithms
| Algorithm | Key Property | Best When |
|-----------|-------------|----------|
| Linear Regression | Simple, interpretable | Linear relationship |
| Ridge | Linear + L2 regularization | Many correlated features |
| Lasso | Linear + L1 regularization | Feature selection needed |
| Decision Tree | Non-linear, interpretable | Capturing interactions |
| Random Forest | Ensemble of trees | General purpose |
| Gradient Boosting | Best accuracy | Competitions, production |

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.datasets import fetch_california_housing

# ── Load California Housing dataset ──────────────────────────────────────────
housing = fetch_california_housing()
X = pd.DataFrame(housing.data, columns=housing.feature_names)
y = housing.target  # Median house value in $100,000s

print("Dataset shape:", X.shape)
print("Target: median house value ($100k)")
print(f"Target range: ${y.min():.2f}k–${y.max():.2f}k (×$100,000)")
print("Features:", list(X.columns))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# ── Compare multiple regressors ────────────────────────────────────────────────
regressors = {
    'Linear Regression':   LinearRegression(),
    'Ridge (L2)':          Ridge(alpha=1.0),
    'Lasso (L1)':          Lasso(alpha=0.01),
    'Decision Tree':       DecisionTreeRegressor(max_depth=5, random_state=42),
    'Random Forest':       RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting':   GradientBoostingRegressor(n_estimators=100, random_state=42),
}

print(f"\n{'Model':<22} {'RMSE':>8} {'MAE':>8} {'R²':>8}")
print("-" * 50)
results = {}
for name, model in regressors.items():
    # Tree models don't need scaling; linear models do
    if 'Tree' in name or 'Forest' in name or 'Boosting' in name:
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
    else:
        model.fit(X_train_sc, y_train)
        preds = model.predict(X_test_sc)
    
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae  = mean_absolute_error(y_test, preds)
    r2   = r2_score(y_test, preds)
    results[name] = {'RMSE': rmse, 'MAE': mae, 'R2': r2}
    print(f"{name:<22} {rmse:>8.4f} {mae:>8.4f} {r2:>8.4f}")

print(f"\nInterpretation: RMSE in $100k units. R²=1.0 is perfect.")
best = min(results, key=lambda k: results[k]['RMSE'])
print(f"Best model: {best} (RMSE=${results[best]['RMSE']*100:.1f}k)")

---
## 4. Supervised Learning: Classification

**Classification** predicts a **category** (spam/not-spam, cancer/benign, dog/cat/bird).

**Analogy**: Classification is sorting mail into boxes — this goes in 'bills', this goes in 'personal', this goes in 'spam'. You're assigning each input to one of a finite set of categories.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                               f1_score, roc_auc_score, classification_report,
                               confusion_matrix)

# ── Breast Cancer: Binary Classification ──────────────────────────────────────
# 30 features from cell nuclei measurements; predict malignant (1) vs benign (0)
cancer = load_breast_cancer()
X_c = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y_c = cancer.target

print("Breast Cancer Dataset:")
print(f"  Shape: {X_c.shape}, Classes: {cancer.target_names}")
print(f"  Class balance: {np.bincount(y_c)} (benign=1, malignant=0)")

X_tr, X_te, y_tr, y_te = train_test_split(X_c, y_c, test_size=0.2, random_state=42, stratify=y_c)

scaler_c = StandardScaler()
X_tr_sc  = scaler_c.fit_transform(X_tr)
X_te_sc  = scaler_c.transform(X_te)

classifiers = {
    'Logistic Regression':  (LogisticRegression(max_iter=1000, random_state=42), True),
    'K-Nearest Neighbors':  (KNeighborsClassifier(n_neighbors=5), True),
    'Decision Tree':        (DecisionTreeClassifier(max_depth=5, random_state=42), False),
    'Random Forest':        (RandomForestClassifier(n_estimators=100, random_state=42), False),
    'Gradient Boosting':    (GradientBoostingClassifier(n_estimators=100, random_state=42), False),
    'SVM (RBF kernel)':     (SVC(kernel='rbf', probability=True, random_state=42), True),
    'Naive Bayes':          (GaussianNB(), False),
}

print(f"\n{'Model':<24} {'Acc':>6} {'Prec':>6} {'Recall':>7} {'F1':>6} {'AUC':>6}")
print("-" * 58)

best_f1 = 0
best_clf = None
for name, (clf, needs_scale) in classifiers.items():
    Xtr = X_tr_sc if needs_scale else X_tr
    Xte = X_te_sc if needs_scale else X_te
    clf.fit(Xtr, y_tr)
    pred  = clf.predict(Xte)
    proba = clf.predict_proba(Xte)[:, 1]
    acc   = accuracy_score(y_te, pred)
    prec  = precision_score(y_te, pred)
    rec   = recall_score(y_te, pred)
    f1    = f1_score(y_te, pred)
    auc   = roc_auc_score(y_te, proba)
    print(f"{name:<24} {acc:>6.3f} {prec:>6.3f} {rec:>7.3f} {f1:>6.3f} {auc:>6.3f}")
    if f1 > best_f1:
        best_f1 = f1
        best_clf = (name, clf, needs_scale)

print(f"\nBest model by F1: {best_clf[0]} (F1={best_f1:.3f})")

---
## 5. Model Evaluation: Metrics & Cross-Validation

### Classification Metrics Explained

**Confusion Matrix**:
```
                Predicted: Pos  Predicted: Neg
Actual: Pos       TP              FN
Actual: Neg       FP              TN
```

- **Accuracy** = (TP+TN)/(TP+TN+FP+FN) — misleading if classes are imbalanced
- **Precision** = TP/(TP+FP) — "Of all predicted positives, how many were actually positive?"
- **Recall** = TP/(TP+FN) — "Of all actual positives, how many did we catch?"
- **F1** = 2×(Precision×Recall)/(Precision+Recall) — harmonic mean, good for imbalanced classes
- **AUC-ROC** = Area under the ROC curve — model's ability to discriminate (1=perfect, 0.5=random)

**When to prioritize**:
- Medical diagnosis: maximize **Recall** (catch all cancers, even if some false alarms)
- Spam filter: maximize **Precision** (don't block real emails, even if some spam gets through)
- Most cases: use **F1** or **AUC-ROC**

### Cross-Validation

A single train-test split is noisy. **Cross-validation** gives a more reliable estimate:
1. Split data into K folds
2. Train on K-1 folds, evaluate on 1 fold
3. Repeat K times (each fold gets a turn as test set)
4. Average the K scores

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import RocCurveDisplay, ConfusionMatrixDisplay

# Use Random Forest on breast cancer
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)

# ── Cross-validation ──────────────────────────────────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(rf_clf, X_c, y_c, cv=cv, scoring='f1')

print("=== 5-Fold Stratified Cross-Validation (Random Forest) ===")
print(f"F1 scores per fold: {cv_scores.round(4)}")
print(f"Mean F1: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

# ── Full classification report ─────────────────────────────────────────────────
rf_clf.fit(X_tr, y_tr)
y_pred_rf = rf_clf.predict(X_te)

print(f"\n=== Classification Report ===")
print(classification_report(y_te, y_pred_rf, target_names=['Malignant', 'Benign']))

# ── Visualization: Confusion Matrix + ROC Curve ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Confusion Matrix
ConfusionMatrixDisplay.from_estimator(
    rf_clf, X_te, y_te,
    display_labels=['Malignant', 'Benign'],
    cmap='Blues', ax=axes[0]
)
axes[0].set_title('Confusion Matrix: Random Forest')

# ROC Curve
RocCurveDisplay.from_estimator(
    rf_clf, X_te, y_te, ax=axes[1], name='Random Forest'
)
axes[1].set_title('ROC Curve (AUC = area under this curve)')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random (AUC=0.5)')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 6. Ensemble Methods

**Ensemble methods** combine multiple models to produce better results than any single model.

**Analogy**: Instead of asking one expert their opinion, you ask 100 experts and take the majority vote (Random Forest). Or you build a chain of experts where each one focuses on fixing the mistakes of the previous (Gradient Boosting).

| Method | How it works | Key Parameter |
|--------|-------------|---------------|
| **Bagging** (Random Forest) | Train many trees on random data subsets; average predictions | `n_estimators`, `max_features` |
| **Boosting** (GradientBoosting) | Train trees sequentially; each focuses on previous errors | `n_estimators`, `learning_rate` |
| **Voting** | Combine different model types | `voting='hard'/'soft'` |
| **Stacking** | Use model predictions as features for a meta-model | `final_estimator` |

In [ ]:
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier, 
                               AdaBoostClassifier, BaggingClassifier,
                               VotingClassifier, StackingClassifier)

X_c_tr_sc = scaler_c.fit_transform(X_tr)
X_c_te_sc = scaler_c.transform(X_te)

# ── Voting Classifier: combine different types of models ───────────────────────
voting_clf = VotingClassifier(
    estimators=[
        ('lr',  LogisticRegression(max_iter=1000, random_state=42)),
        ('rf',  RandomForestClassifier(n_estimators=50, random_state=42)),
        ('svm', SVC(probability=True, random_state=42))
    ],
    voting='soft'  # 'soft': average probabilities (better than 'hard': majority vote)
)
voting_clf.fit(X_c_tr_sc, y_tr)
print(f"Voting Classifier F1: {f1_score(y_te, voting_clf.predict(X_c_te_sc)):.4f}")

# ── Stacking: use model predictions as features for a meta-model ──────────────
stacking_clf = StackingClassifier(
    estimators=[
        ('lr',  LogisticRegression(max_iter=1000)),
        ('rf',  RandomForestClassifier(n_estimators=50, random_state=42)),
        ('gb',  GradientBoostingClassifier(n_estimators=50, random_state=42))
    ],
    final_estimator=LogisticRegression(),  # Meta-learner
    cv=5
)
stacking_clf.fit(X_tr, y_tr)  # Note: unscaled — stacking handles it internally
print(f"Stacking Classifier F1:  {f1_score(y_te, stacking_clf.predict(X_te)):.4f}")

# ── Gradient Boosting with learning curve ─────────────────────────────────────
from sklearn.model_selection import learning_curve

gb = GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=3, random_state=42)

train_sizes, train_scores, val_scores = learning_curve(
    gb, X_c, y_c, cv=5, scoring='f1',
    train_sizes=np.linspace(0.1, 1.0, 10)
)

plt.figure(figsize=(8, 4))
plt.plot(train_sizes, train_scores.mean(axis=1), 'b-o', label='Training F1')
plt.plot(train_sizes, val_scores.mean(axis=1), 'r-o', label='Validation F1')
plt.fill_between(train_sizes,
                  train_scores.mean(1) - train_scores.std(1),
                  train_scores.mean(1) + train_scores.std(1), alpha=0.15)
plt.fill_between(train_sizes,
                  val_scores.mean(1) - val_scores.std(1),
                  val_scores.mean(1) + val_scores.std(1), alpha=0.15, color='red')
plt.title('Learning Curve: Gradient Boosting')
plt.xlabel('Training Set Size')
plt.ylabel('F1 Score')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()
print("Learning Curve: If train>>val → overfitting. If both low → underfitting.")

---
## 7. Unsupervised Learning: Clustering & Dimensionality Reduction

**Unsupervised learning**: no labels! The model finds patterns in data on its own.

### Clustering: Finding Natural Groups
**K-Means**: Assigns each data point to the nearest centroid, then updates centroids. Repeat until stable.
**DBSCAN**: Finds clusters based on density — great for irregular shapes and outlier detection.

### PCA (Principal Component Analysis): Finding Structure
PCA compresses high-dimensional data into fewer dimensions while preserving maximum variance. It's used for: visualization, noise reduction, feature engineering.

In [ ]:
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# ── K-Means Clustering ─────────────────────────────────────────────────────────
# Use iris features (no labels)
X_iris, y_iris = load_iris(return_X_y=True)

# Find optimal K using elbow method
inertias    = []
silhouettes = []
K_range = range(2, 10)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_iris)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_iris, km.labels_))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Elbow plot
axes[0].plot(K_range, inertias, 'bo-', markersize=8)
axes[0].axvline(3, color='red', linestyle='--', label='K=3 (true classes)')
axes[0].set_title('Elbow Method: Choose K')
axes[0].set_xlabel('K (number of clusters)')
axes[0].set_ylabel('Inertia (within-cluster sum of squares)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Silhouette scores
axes[1].plot(K_range, silhouettes, 'go-', markersize=8)
best_k = K_range[np.argmax(silhouettes)]
axes[1].axvline(best_k, color='red', linestyle='--', label=f'Best K={best_k}')
axes[1].set_title('Silhouette Score (higher = better separated clusters)')
axes[1].set_xlabel('K')
axes[1].set_ylabel('Silhouette Score')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# PCA visualization of clusters
km_final = KMeans(n_clusters=3, random_state=42, n_init=10)
km_labels = km_final.fit_predict(X_iris)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_iris)

scatter = axes[2].scatter(X_pca[:, 0], X_pca[:, 1], c=km_labels, 
                            cmap='Set1', alpha=0.7, s=40)
axes[2].scatter(pca.transform(km_final.cluster_centers_)[:, 0],
                 pca.transform(km_final.cluster_centers_)[:, 1],
                 marker='*', s=300, c='black', zorder=5, label='Centroids')
axes[2].set_title(f'K-Means (K=3) on PCA-reduced Iris')
axes[2].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
axes[2].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
axes[2].legend()

plt.tight_layout()
plt.show()

print(f"PCA: 2 components explain {pca.explained_variance_ratio_.sum():.1%} of variance")

# ── PCA for High-Dimensional Data ─────────────────────────────────────────────
# How many components to keep?
pca_full = PCA().fit(X_c.values)  # Breast cancer: 30 features
cumvar = np.cumsum(pca_full.explained_variance_ratio_)
n_95  = np.argmax(cumvar >= 0.95) + 1
print(f"\nBreast Cancer (30 features):")
print(f"Components needed for 95% variance: {n_95} (vs original 30)")
print(f"That's a {(1-n_95/30)*100:.0f}% dimensionality reduction!")

---
## 8. Pipelines: The Professional Pattern

A `Pipeline` chains preprocessing steps + model into a **single estimator object**. This is how production ML code is written.

**Benefits**:
1. **No data leakage**: fitting happens correctly inside CV folds
2. **Clean code**: one object for the whole workflow
3. **Easy deployment**: `pipeline.predict(raw_X)` — no manual preprocessing
4. **Safe hyperparameter search**: tune preprocessing AND model parameters together

**Analogy**: A pipeline is an assembly line — raw material goes in one end, finished product comes out the other. No manual hand-off between stations.

In [ ]:
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# ── Mixed dataset: numeric + categorical features ──────────────────────────────
np.random.seed(42)
n = 500
mixed_df = pd.DataFrame({
    'age':       np.random.randint(20, 65, n).astype(float),
    'income':    np.random.exponential(50000, n),
    'debt_ratio':np.random.uniform(0, 0.8, n),
    'city':      np.random.choice(['NYC', 'LA', 'Chicago', 'Houston'], n),
    'job_type':  np.random.choice(['Employed', 'Self-Employed', 'Unemployed'], n,
                                   p=[0.6, 0.25, 0.15]),
})
# Introduce missing values
mixed_df.loc[mixed_df.sample(30, random_state=1).index, 'age'] = np.nan
mixed_df.loc[mixed_df.sample(20, random_state=2).index, 'city'] = np.nan

# Target: loan approval (logistic function of features)
score = (mixed_df['income'].fillna(0)/100000 - mixed_df['debt_ratio'].fillna(0.5) 
         + (mixed_df['job_type'].fillna('Unemployed') == 'Employed').astype(float) * 0.5
         + np.random.randn(n) * 0.3)
y_loan = (score > score.median()).astype(int)

X_loan = mixed_df
X_loan_tr, X_loan_te, y_loan_tr, y_loan_te = train_test_split(
    X_loan, y_loan, test_size=0.2, random_state=42
)

# ── ColumnTransformer: different preprocessing for different columns ───────────
numeric_features    = ['age', 'income', 'debt_ratio']
categorical_features = ['city', 'job_type']

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),  # Fill missing with median
    ('scaler',  StandardScaler())                   # Then scale
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),  # Fill missing with mode
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline,    numeric_features),
    ('cat', categorical_pipeline, categorical_features),
])

# ── Full ML Pipeline ───────────────────────────────────────────────────────────
full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier',   RandomForestClassifier(n_estimators=100, random_state=42))
])

# Train and evaluate: raw DataFrame → Pipeline handles EVERYTHING
full_pipeline.fit(X_loan_tr, y_loan_tr)
y_loan_pred = full_pipeline.predict(X_loan_te)

print("=== Pipeline: Raw Data → Prediction ===")
print(f"Accuracy:  {accuracy_score(y_loan_te, y_loan_pred):.4f}")
print(f"F1 Score:  {f1_score(y_loan_te, y_loan_pred):.4f}")
print(f"AUC-ROC:   {roc_auc_score(y_loan_te, full_pipeline.predict_proba(X_loan_te)[:,1]):.4f}")

# The pipeline can predict on new raw data (with missing values!) directly
new_applicant = pd.DataFrame({
    'age': [35], 'income': [75000], 'debt_ratio': [0.3],
    'city': ['NYC'], 'job_type': ['Employed']
})
approval_prob = full_pipeline.predict_proba(new_applicant)[0, 1]
print(f"\nNew applicant approval probability: {approval_prob:.1%}")

---
## 9. Hyperparameter Tuning

**Hyperparameters** are model settings you choose before training (like `n_estimators=100` for Random Forest). They're not learned from data.

| Search Method | How | When |
|--------------|-----|------|
| `GridSearchCV` | Try all combinations | Small param grid |
| `RandomizedSearchCV` | Try random subset | Large param space |
| `HalvingRandomSearchCV` | Successively eliminate bad params | Efficient large search |
| `BayesSearchCV` (optuna) | Bayesian optimization | Production tuning |

In [ ]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

# ── GridSearchCV ──────────────────────────────────────────────────────────────
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth':    [None, 5, 10],
    'min_samples_split': [2, 5],
}  # 3 × 3 × 2 = 18 combinations × 5 CV folds = 90 fits!

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,      # Use all CPU cores
    verbose=0       # Set to 1 or 2 for progress output
)

print("Running GridSearchCV (18 param combinations × 5 CV folds)...")
grid_search.fit(X_tr, y_tr)

print(f"Best params:     {grid_search.best_params_}")
print(f"Best CV F1:      {grid_search.best_score_:.4f}")
best_model = grid_search.best_estimator_
print(f"Test F1:         {f1_score(y_te, best_model.predict(X_te)):.4f}")

# ── RandomizedSearchCV for large search spaces ─────────────────────────────────
from scipy.stats import randint, uniform

param_dist = {
    'n_estimators':     randint(50, 500),
    'max_depth':        [None, 3, 5, 10, 15, 20],
    'min_samples_split':randint(2, 20),
    'min_samples_leaf': randint(1, 10),
    'max_features':     ['sqrt', 'log2', None]
}  # Vast search space — random sampling is much more efficient!

rand_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_dist,
    n_iter=30,      # Try 30 random combinations (vs all 1000s for grid)
    cv=5,
    scoring='f1',
    n_jobs=-1,
    random_state=42
)
print("\nRunning RandomizedSearchCV (30 random combinations)...")
rand_search.fit(X_tr, y_tr)
print(f"Best params:  {rand_search.best_params_}")
print(f"Best CV F1:   {rand_search.best_score_:.4f}")
print(f"Test F1:      {f1_score(y_te, rand_search.best_estimator_.predict(X_te)):.4f}")

---
## 10. Feature Selection & Importance

Not all features contribute equally. Removing useless features can:
- Reduce overfitting
- Speed up training
- Improve interpretability
- Reduce data collection costs

In [ ]:
from sklearn.feature_selection import (SelectKBest, f_classif, mutual_info_classif,
                                        RFE, SelectFromModel)
from sklearn.inspection import permutation_importance

# ── Feature Importances from Random Forest ─────────────────────────────────────
rf_final = RandomForestClassifier(n_estimators=100, random_state=42)
rf_final.fit(X_tr, y_tr)

feature_names = list(X_c.columns)
importances   = rf_final.feature_importances_
sorted_idx    = np.argsort(importances)[::-1][:15]  # Top 15 features

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of importances
axes[0].barh([feature_names[i] for i in sorted_idx[::-1]],
              importances[sorted_idx[::-1]],
              color='steelblue', alpha=0.8)
axes[0].set_title('Feature Importances: Random Forest\n(Mean Decrease in Impurity)')
axes[0].set_xlabel('Importance')

# Permutation importance (more reliable, model-agnostic)
perm_imp = permutation_importance(rf_final, X_te, y_te, 
                                   n_repeats=10, random_state=42, n_jobs=-1)
perm_idx = np.argsort(perm_imp.importances_mean)[::-1][:15]

axes[1].boxplot(
    [perm_imp.importances[perm_idx[i]] for i in range(15)],
    labels=[feature_names[perm_idx[i]] for i in range(15)],
    vert=False
)
axes[1].set_title('Permutation Importance\n(Impact on test F1 when feature is shuffled)')
axes[1].set_xlabel('Mean Accuracy Decrease')

plt.tight_layout()
plt.show()

# ── SelectFromModel: automatically select features above threshold ──────────────
selector = SelectFromModel(rf_final, threshold='mean')  # Keep features above avg importance
X_tr_sel = selector.transform(X_tr)
X_te_sel = selector.transform(X_te)
n_selected = X_tr_sel.shape[1]
print(f"\nFeatures selected: {n_selected}/{X_tr.shape[1]}")
print(f"Selected features: {[feature_names[i] for i in selector.get_support(indices=True)[:5]]}...")

# Train model on selected features only
rf_sel = RandomForestClassifier(n_estimators=100, random_state=42)
rf_sel.fit(X_tr_sel, y_tr)
print(f"F1 with all {X_tr.shape[1]} features: {f1_score(y_te, rf_final.predict(X_te)):.4f}")
print(f"F1 with {n_selected} features:        {f1_score(y_te, rf_sel.predict(X_te_sel)):.4f}")

---
## 11. Common Pitfalls

| Pitfall | Why It's Dangerous | Fix |
|---------|-------------------|-----|
| **Data leakage** | Preprocessing fitted on all data → falsely optimistic eval | Fit preprocessors ONLY on training data; use Pipelines |
| **Not stratifying** | Imbalanced classes → test set might have no rare class | `train_test_split(..., stratify=y)` |
| **Using accuracy with imbalanced classes** | 95% accuracy looks great when 95% are majority class | Use F1, AUC-ROC, or precision-recall |
| **Overfitting to validation set** | Tuning on val set makes it a training set | Use a separate held-out test set NEVER used in tuning |
| **Scaling before split** | StandardScaler fitted on all data leaks test stats | Split FIRST, then scale (or use Pipeline) |
| **Ignoring class imbalance** | Minority class predicted poorly | Use `class_weight='balanced'`, SMOTE, or adjust threshold |
| **Tree depth too large** | Overfits to training noise | Set `max_depth`, `min_samples_split`; use CV |
| **Assuming more data = better** | Collecting noise helps nothing | Clean data > more data |

---
## 12. Mini Project: End-to-End ML Pipeline

**Scenario**: You work at a bank. Given customer features, predict whether they will default on a loan (binary classification). This requires:
1. Handling mixed types (numeric + categorical)
2. Dealing with missing values
3. Handling class imbalance (most people don't default)
4. Selecting the best model with cross-validation
5. Tuning hyperparameters
6. Evaluating with appropriate metrics
7. Explaining predictions with feature importance

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
import numpy as np, pandas as pd, matplotlib.pyplot as plt

np.random.seed(42)

# ── Generate synthetic loan data ───────────────────────────────────────────────
n = 2000
df = pd.DataFrame({
    'age':          np.random.randint(22, 70, n).astype(float),
    'income':       np.random.exponential(55000, n),
    'loan_amount':  np.random.uniform(1000, 100000, n),
    'credit_score': np.random.normal(650, 80, n).clip(300, 850),
    'debt_ratio':   np.random.uniform(0.05, 0.85, n),
    'months_employed': np.random.randint(0, 360, n).astype(float),
    'num_credit_lines': np.random.randint(0, 15, n),
    'education':    np.random.choice(['High School', 'Bachelor', 'Master', 'PhD'], n,
                                      p=[0.3, 0.4, 0.2, 0.1]),
    'employment_type': np.random.choice(['Full-Time', 'Part-Time', 'Self-Employed', 'Unemployed'], n,
                                         p=[0.55, 0.15, 0.20, 0.10]),
    'has_mortgage': np.random.choice([0, 1], n, p=[0.4, 0.6]),
})

# Add missing values (realistic)
for col, pct in [('age', 0.03), ('income', 0.05), ('education', 0.08)]:
    idx = df.sample(int(n*pct), random_state=hash(col)%100).index
    df.loc[idx, col] = np.nan

# Target: default probability (realistic function of features)
z = (-df['credit_score'].fillna(650)/200 +
     df['debt_ratio'].fillna(0.4)*3 +
     (df['employment_type'].fillna('Unemployed') == 'Unemployed').astype(float)*2 +
     (df['employment_type'].fillna('Unemployed') == 'Part-Time').astype(float)*0.5 -
     df['income'].fillna(50000)/100000 +
     np.random.randn(n) * 0.5)

default_prob = 1 / (1 + np.exp(-z))
y_default = (default_prob > 0.5).astype(int)
print(f"Default rate: {y_default.mean():.1%} (target: ~15-25% for realism)")

# ── Split ─────────────────────────────────────────────────────────────────────
X_df, X_hold, y_df, y_hold = train_test_split(
    df, y_default, test_size=0.15, random_state=42, stratify=y_default
)
X_tr2, X_te2, y_tr2, y_te2 = train_test_split(
    X_df, y_df, test_size=0.2, random_state=42, stratify=y_df
)

# ── Define feature groups ──────────────────────────────────────────────────────
numeric_cols     = ['age', 'income', 'loan_amount', 'credit_score', 'debt_ratio',
                    'months_employed', 'num_credit_lines', 'has_mortgage']
categorical_cols = ['education', 'employment_type']

# ── Build Pipeline ─────────────────────────────────────────────────────────────
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe',     OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer([
    ('num', num_pipe, numeric_cols),
    ('cat', cat_pipe, categorical_cols)
])

model_pipeline = Pipeline([
    ('prep',  preprocessor),
    ('model', GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.05,
        max_depth=4, min_samples_split=10,
        subsample=0.8, random_state=42
    ))
])

# ── Cross-validate first ───────────────────────────────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_aucs = cross_val_score(model_pipeline, X_tr2, y_tr2, cv=cv, scoring='roc_auc', n_jobs=-1)
print(f"\n5-Fold CV AUC: {cv_aucs.mean():.4f} ± {cv_aucs.std():.4f}")

# ── Fit and evaluate ──────────────────────────────────────────────────────────
model_pipeline.fit(X_tr2, y_tr2)
y_proba = model_pipeline.predict_proba(X_te2)[:, 1]
y_pred2 = (y_proba > 0.5).astype(int)

print(f"Test AUC-ROC:  {roc_auc_score(y_te2, y_proba):.4f}")
print(f"\nClassification Report:")
print(classification_report(y_te2, y_pred2, target_names=['No Default', 'Default']))

# ── Hold-out evaluation (truly unseen data) ────────────────────────────────────
y_hold_proba = model_pipeline.predict_proba(X_hold)[:, 1]
print(f"Hold-out AUC:  {roc_auc_score(y_hold, y_hold_proba):.4f}  ← truly unseen data")

# ── Feature Importance ─────────────────────────────────────────────────────────
model_fitted = model_pipeline.named_steps['model']
ohe_cats     = model_pipeline.named_steps['prep'].named_transformers_['cat'].named_steps['ohe'].get_feature_names_out(categorical_cols)
all_features = numeric_cols + list(ohe_cats)
imp_series   = pd.Series(model_fitted.feature_importances_, index=all_features).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
imp_series.head(12).plot(kind='barh', ax=axes[0], color='steelblue', alpha=0.8)
axes[0].invert_yaxis()
axes[0].set_title('Feature Importances: Loan Default Model')

ConfusionMatrixDisplay.from_predictions(y_te2, y_pred2, 
    display_labels=['No Default', 'Default'], cmap='Blues', ax=axes[1])
axes[1].set_title('Confusion Matrix: Test Set')

plt.tight_layout()
plt.show()

print("\n" + "="*50)
print("LOAN DEFAULT PREDICTION MODEL — SUMMARY")
print("="*50)
print(f"Training data: {len(X_tr2):,} customers")
print(f"Test set AUC:  {roc_auc_score(y_te2, y_proba):.3f}")
print(f"Top risk factors: {', '.join(imp_series.head(3).index.tolist())}")
print(f"Model: GradientBoostingClassifier in a full Pipeline")

---
## 13. Interview Q&A

**Q1: What is the difference between overfitting and underfitting?**  
**A**: Overfitting: the model learned the training data too well (including noise) — high train accuracy, low test accuracy. Underfitting: the model is too simple — low accuracy on both train and test. The goal is the "Goldilocks zone" — complex enough to learn patterns, simple enough to generalize. Diagnose with a learning curve: overfitting shows a large gap between train and val scores; underfitting shows both low.

---
**Q2: What is data leakage and why is it dangerous?**  
**A**: Data leakage is when information from the test set (or future data) accidentally influences training. Example: fitting a scaler on all data before splitting — the scaler's mean and std incorporate test data, so the model indirectly "knows" something about the test set. It causes falsely optimistic validation scores that don't generalize to production. Fix: always split first, use Pipelines for correct CV.

---
**Q3: What is cross-validation and why use it instead of a single train-test split?**  
**A**: Cross-validation (especially k-fold) uses all data for both training and validation across multiple rounds. Benefits: (1) more reliable performance estimate — less sensitive to the random split, (2) all data is eventually used for training, (3) gives a standard deviation of performance (uncertainty estimate). A single split can be misleading if you happen to get an easy or hard test set.

---
**Q4: What is the difference between precision and recall? When to prioritize each?**  
**A**: Precision = of all positive predictions, what fraction were actually positive (minimize false alarms). Recall = of all actual positives, what fraction did we catch (minimize missed cases). Medical screening: maximize recall (don't miss cancer). Email spam filter: maximize precision (don't send real email to spam). In practice, F1 = harmonic mean of both — use F1 when you want a balance.

---
**Q5: What is the difference between Random Forest and Gradient Boosting?**  
**A**: Both are ensembles of decision trees. Random Forest builds trees in **parallel** on random data subsets and averages predictions (bagging). Gradient Boosting builds trees **sequentially** — each tree focuses on the residual errors of the previous ones. GBoost typically achieves higher accuracy but trains slower and is more prone to overfitting. RF is more robust and faster.

---
**Q6: What is a sklearn Pipeline and why should you use it?**  
**A**: A Pipeline chains preprocessing steps and a model into one estimator. It prevents data leakage (fits preprocessing only on training data inside each CV fold), enables clean deployment (raw data → prediction), and allows hyperparameter tuning of both preprocessing AND model parameters simultaneously with `GridSearchCV`.

---
**Q7: When would you use KNN vs Logistic Regression vs Random Forest?**  
**A**: KNN: simple, good for small datasets, non-linear boundaries — but slow at prediction time (O(n)). Logistic Regression: interpretable, fast, good baseline — but assumes linear decision boundary. Random Forest: handles non-linearity, robust to outliers, works well with mixed features, low hyperparameter tuning needed — good general-purpose choice.

---
## 14. Resources

### Official
- **Documentation**: https://scikit-learn.org/stable/
- **User Guide** (thorough): https://scikit-learn.org/stable/user_guide.html
- **API Reference**: https://scikit-learn.org/stable/modules/classes.html
- **Example Gallery**: https://scikit-learn.org/stable/auto_examples/

### YouTube
- **StatQuest with Josh Starmer** (best ML theory): https://www.youtube.com/c/joshstarmer
  - Recommended: "Machine Learning" playlist
- **Corey Schafer — Scikit-Learn**: https://www.youtube.com/playlist?list=PL-osiE80TeTd2vbCRGJrRhIiSEIYPF_Ij
- **Sentdex — Machine Learning**: https://www.youtube.com/playlist?list=PLQVvvaa0QuDfKTOs3Keq_kaG2P55YRn5v

### Textbooks (Free)
- **Introduction to Statistical Learning (ISLR)** — https://www.statlearning.com/ (THE reference)
- **Hands-On Machine Learning (Geron)** — Chapter 2-6 covers sklearn deeply
- **Pattern Recognition & Machine Learning (Bishop)** — Advanced theory

### Papers
- **Scikit-learn Paper**: Pedregosa et al. (2011). *Journal of Machine Learning Research*, 12, 2825-2830. https://jmlr.csail.mit.edu/papers/v12/pedregosa11a.html

---
## Summary & What's Next

### What You Learned
| Concept | Key Point |
|---------|----------|
| API Contract | Every estimator: `.fit()`, `.predict()`, `.score()`, `.transform()` |
| Preprocessing | Scale after splitting; use Pipeline to prevent leakage |
| Regression | Linear→Ridge/Lasso for regularization; RF/GBoost for non-linear |
| Classification | Logistic Regression baseline → RF/GBoost for production |
| Evaluation | Accuracy misleads; use F1/AUC for imbalanced; always cross-validate |
| Ensemble | RF=parallel bagging; GBoost=sequential boosting |
| Unsupervised | KMeans clustering; PCA for dimensionality reduction |
| Pipelines | Chain preprocessing+model → correct CV, easy deployment |
| Tuning | GridSearch (small grids), RandomizedSearch (large spaces) |
| Feature Importance | RF importance + permutation importance + SelectFromModel |

### What's Next?
- **Next Notebook**: XGBoost — the model that wins Kaggle competitions
- **Practice**: Load a Kaggle dataset and run the complete pipeline from this notebook
- **Challenge**: Implement a stacking ensemble with 5+ base models on the California Housing dataset and beat Gradient Boosting

> **Key insight**: Scikit-Learn's consistent API is a superpower. Once you know the `.fit()/.predict()/.score()` pattern, you can try any algorithm in 2 lines. The Pipeline pattern prevents data leakage and is how professional ML code is written.